# Merging the Kelly Dataset with CRSP Returns

## What this notebook does and why it exists

The Kelly, Pruitt & Su (2020) dataset (`datashare.csv`) contains 94 pre-computed cross-sectional signals for US stocks from 1957 to 2021. It is the foundation of our training data. However, it does not contain usable return data for our purposes — the return column in the raw Kelly file follows a specific timing convention that needs to be aligned with our own CRSP return pull.

Our CRSP pull (`crsp_returns.csv`) contains cleaned, winsorised monthly returns with delisting adjustments applied. This is the return series we want as our prediction target.

The goal of this notebook is simple: attach our CRSP returns to the Kelly signal rows, so that every row has both the signals observed at time t and the return earned in month t+1. The result is `kelly_final.csv`, which is the master input for the entire backtest pipeline.

## The critical timing convention

The single most important thing to understand here is the **n+1 date shift**.

In the Kelly dataset, the return on row `(permno, date=2020-01)` is the return **earned in February 2020** — the next month. The signals on that row are observed at the end of January 2020, and the portfolio formed at that point earns its return over February 2020.

In CRSP, the return on `date=2020-02` is also the return earned **during** February 2020.

So: `Kelly row date=2020-01` matches `CRSP row date=2020-02`.

This shift is already baked into the CRSP pull — we shifted returns back by one month when cleaning CRSP. So a simple join on `(permno, date)` works correctly. But it must be verified, and we do that at the end of this notebook.

## Step 1 — Load both datasets and verify alignment

Before merging anything, I want to confirm that the two datasets are talking about the same universe of stocks and that the dates are formatted identically. The Kelly dataset uses an integer date format (`YYYYMMDD`) which needs to be converted to the same month-start timestamp format that CRSP uses.

I also check the permno overlap upfront. There are 428 permnos in Kelly but not in CRSP — these are stocks that appear in the Kelly signal dataset but whose return history is not in our CRSP pull. This is expected: some stocks in the original Kelly universe were delisted before our CRSP filter period, or failed our share code / price filters. These rows will end up with missing returns after the merge and will be dropped in the final step.

In [2]:
import pandas as pd
import numpy as np

# Load the raw Kelly signal dataset and our cleaned CRSP return file
kelly = pd.read_csv('datashare\\datashare.csv', low_memory=True)
crsp  = pd.read_csv('crsp_returns.csv', parse_dates=['date'])

# The Kelly DATE column is an integer in YYYYMMDD format (e.g. 20200131)
# We need to convert it to a month-start timestamp to match CRSP's date format
# We go via period to safely land on the first of the month regardless of the day
kelly['date'] = pd.to_datetime(
    kelly['DATE'].astype(str), format='%Y%m%d'
)
kelly['date'] = kelly['date'].dt.to_period('M').dt.to_timestamp()

print("Kelly date range:", kelly['date'].min(), "to", kelly['date'].max())
print("CRSP date range:",  crsp['date'].min(),  "to", crsp['date'].max())
print("Kelly rows:", len(kelly))
print("CRSP rows:",  len(crsp))

# Check how much of the Kelly universe we can actually match to CRSP
# Any permno in Kelly but not CRSP will produce a missing return after the merge
kelly_permnos = set(kelly['permno'].unique())
crsp_permnos  = set(crsp['permno'].unique())

print(f"\nKelly unique permnos: {len(kelly_permnos)}")
print(f"CRSP unique permnos:  {len(crsp_permnos)}")
print(f"In both:              {len(kelly_permnos & crsp_permnos)}")
print(f"In Kelly but not CRSP: {len(kelly_permnos - crsp_permnos)}")

## Step 2 — Merge CRSP returns onto Kelly

I only bring three columns from CRSP into the Kelly dataset: `ret` (the cleaned winsorised return that is our prediction target), `ret_raw` (the pre-winsorisation return, kept for diagnostics), and `me` (market equity in millions, needed for the AUM capacity analysis and for size-related features).

Everything else — all 94 signals — already lives in Kelly. There is no point duplicating data.

I use a **left merge** on `(permno, date)`. This means every Kelly row is preserved regardless of whether a matching CRSP return exists. Rows with no CRSP match will have `ret = NaN`. These get filtered out in the final save step. Using a left merge here (rather than an inner merge that would silently drop rows) gives me full visibility into exactly how many rows fail to match and why.

In [3]:
# Only pull the columns we actually need from CRSP — ret, ret_raw, and market equity
# Everything else (all 94 signals) already comes from the Kelly dataset
crsp_to_merge = crsp[['permno', 'date', 'ret', 'ret_raw', 'me']].copy()

# Left merge: every Kelly row is kept, CRSP columns are NaN where no match exists
# This lets me see exactly how many rows fail to match rather than silently dropping them
kelly_with_ret = kelly.merge(
    crsp_to_merge,
    on=['permno', 'date'],
    how='left'
)

print(f"\nAfter merge:")
print(f"  Rows: {len(kelly_with_ret)}")
print(f"  Rows with ret: {kelly_with_ret['ret'].notna().sum():,}")
print(f"  Rows missing ret: {kelly_with_ret['ret'].isna().sum():,}")
print(f"  Match rate: {kelly_with_ret['ret'].notna().mean():.2%}")

## Step 3 — Validate the match rate

A 94.61% match rate is acceptable and expected. The missing 5.39% are stocks that appear in the Kelly signal universe but whose returns are not in our cleaned CRSP file — typically because they were delisted before our observation window, failed our price or share code filters, or were on exchanges outside NYSE/AMEX/NASDAQ.

The guardrail here is the 80% threshold. If the match rate drops below 80%, something is fundamentally wrong with the date alignment — the most common cause would be a timezone or date format mismatch between the two datasets. In that case the code triggers a diagnostic that cross-references specific permnos between the two files to identify where the misalignment is occurring.

Since we are above 80%, we move on.

In [4]:
# A match rate above 80% confirms the date alignment worked correctly
# Below 80% would indicate a systematic mismatch — wrong date format, timezone issue, etc.
if kelly_with_ret['ret'].notna().mean() < 0.80:
    print("\nWARNING: Match rate below 80% — checking date alignment...")
    
    # Pull the unmatched rows and look at a sample to understand the pattern
    missing = kelly_with_ret[kelly_with_ret['ret'].isna()]
    print("\nSample missing rows:")
    print(missing[['permno','date']].head(10))
    
    # For the first 5 unmatched permnos, check whether they exist in CRSP at all
    # and compare the date ranges — this isolates whether it is a universe issue
    # or a date format issue
    sample_permnos = missing['permno'].head(5).values
    for p in sample_permnos:
        crsp_dates  = crsp[crsp['permno'] == p]['date']
        kelly_dates = kelly_with_ret[kelly_with_ret['permno'] == p]['date']
        print(f"\npermno {p}:")
        print(f"  CRSP dates:  {crsp_dates.min()} to {crsp_dates.max()}")
        print(f"  Kelly dates: {kelly_dates.min()} to {kelly_dates.max()}")
else:
    print("\nMatch rate looks good.")

## Step 4 — Intermediate save

I save the full merged dataset (including rows with missing returns) as `kelly_with_returns.csv` before doing any further filtering. This gives me a recoverable checkpoint — if something goes wrong in the cleaning steps below, I do not have to redo the merge from scratch.

The final clean file (`kelly_final.csv`, produced in Step 6) will drop the missing-return rows. But keeping this intermediate file means I can always come back and inspect the full merge output.

In [5]:
# Save the full merged dataset before filtering — this is a recoverable checkpoint
# kelly_final.csv (produced below) will be the clean version with missing rows dropped
kelly_with_ret.to_csv('kelly_with_returns.csv', index=False)
print("\nSaved: kelly_with_returns.csv")
print(f"Columns: {kelly_with_ret.columns.tolist()}")

## Step 5 — Diagnostic checks on the merged data

Before writing the final clean file, I run three diagnostic checks to confirm the merge produced sensible results.

**Check 1 — Where are the missing returns concentrated?**  
Missing returns should be distributed across decades and concentrated in early years (1960s-1980s) where CRSP coverage was thinner. A spike in missing returns in recent years (2010+) would be a red flag indicating something went wrong in the CRSP pull.

**Check 2 — Spot check matched rows**  
I pull 10 random matched rows and visually verify that the return values look like plausible monthly stock returns — roughly in the range of -50% to +50%, not zero, not 100% of the time. This catches cases where the merge matched on the wrong rows.

**Check 3 — Date range of matched rows**  
The matched rows should span 1957 to 2021, matching the Kelly dataset's full history. If the matched range is truncated it means some dates failed to align.

In [6]:
# Check 1: where are the missing returns concentrated by decade?
# We expect more missing in early decades where CRSP coverage was thinner
# A spike in 2010+ would be a red flag
missing = kelly_with_ret[kelly_with_ret['ret'].isna()]

print("Missing by decade:")
missing['year'] = missing['date'].dt.year
print(missing.groupby((missing['year']//10)*10).size())

print("\nMissing by exchange:")
# Checking the year distribution of missing rows — if it's front-loaded (1960s-1980s)
# that is expected. If it's recent, something went wrong.
print(missing['date'].dt.year.value_counts().sort_index().head(20))

# Check 2: spot check 10 random matched rows — do the return values look plausible?
print("\nSample of matched rows (ret column):")
matched = kelly_with_ret[kelly_with_ret['ret'].notna()]
print(matched[['permno','date','ret']].sample(10, random_state=42))

# Check 3: confirm the return distribution looks right
# Should be centred near zero with fat tails — typical of monthly stock returns
print("\nReturn distribution in merged dataset:")
print(kelly_with_ret['ret'].describe())

# Check 3b: confirm date range spans the full Kelly history
print("\nDate range of matched rows:")
print(matched['date'].min(), "to", matched['date'].max())

## Step 6 — Write the final clean file in chunks

The full merged dataset with missing rows is approximately 2GB in memory. Writing it out as a single `.to_csv()` call would require loading a second full copy into memory — doubling peak RAM usage to ~4GB which risks crashing the kernel.

Instead I iterate through the dataframe in 500,000-row chunks, filtering out missing-return rows in each chunk before writing. The first chunk writes the header; subsequent chunks append without headers.

At the same time I drop the original `DATE` column (the raw YYYYMMDD integer from Kelly) since we already have the properly formatted `date` column. No point storing both.

The output `kelly_final.csv` is the master training file for the entire backtest — 3,895,198 rows, 1957 to 2021, all with valid returns.

In [8]:
# Writing in chunks to avoid loading a second full copy of the dataset into memory
# At ~2GB, doing a full .copy() then .to_csv() would push peak RAM above 4GB

output_file = 'kelly_final.csv'
chunk_size  = 500000  # process 500k rows at a time

# Drop the original raw DATE column from Kelly — we already have the clean 'date' column
cols_to_keep = [c for c in kelly_with_ret.columns if c != 'DATE']

first_chunk   = True
total_written = 0

for start in range(0, len(kelly_with_ret), chunk_size):
    chunk = kelly_with_ret.iloc[start:start+chunk_size]
    
    # Drop rows with missing returns — these are stocks outside our CRSP universe
    chunk = chunk[chunk['ret'].notna()]
    chunk = chunk[cols_to_keep]
    
    # Write header on the first chunk only, then append for all subsequent chunks
    chunk.to_csv(output_file,
                 mode='w' if first_chunk else 'a',
                 header=first_chunk,
                 index=False)
    
    total_written += len(chunk)
    first_chunk    = False
    print(f"Written {total_written:,} rows...")

print(f"\nDone. Total rows written: {total_written:,}")
print("Saved: kelly_final.csv")

# Free memory now that we are done — the merged dataframe is no longer needed
del kelly_with_ret
del kelly
del crsp

import gc
gc.collect()
print("Memory freed.")

## Step 7 — Final verification of kelly_final.csv

Rather than loading the full 3.9M-row file into memory just to check it, I verify it lightly: read the first 5 rows to confirm the column structure and return values look right, count total rows using a line count (faster than pandas for a file this size), and scan through in chunks to confirm the date range.

Expected output:
- 3,895,198 rows
- Date range: 1957-01-01 to 2021-12-01
- All 94 Kelly signal columns present plus `date`, `ret`, `ret_raw`, `me`

In [1]:
import pandas as pd

# Read just the first 5 rows to confirm column structure and that values look right
# No need to load 3.9M rows just to verify the file was written correctly
sample = pd.read_csv('kelly_final.csv', nrows=5)
print("Columns:", sample.columns.tolist())
print("Sample:\n", sample[['permno','date','ret']].head())

# Count total rows using a line count — much faster than loading into pandas
row_count = sum(1 for _ in open('kelly_final.csv')) - 1  # subtract 1 for the header
print(f"\nTotal rows: {row_count:,}")

# Scan through in chunks to find the date range without loading everything at once
dates = []
for chunk in pd.read_csv('kelly_final.csv',
                          usecols=['date'],
                          chunksize=500000):
    dates.append(chunk['date'].min())
    dates.append(chunk['date'].max())

print(f"Date range: {min(dates)} to {max(dates)}")